In [2]:
import pandas as pd

# -------------------------------------------------------------
# 1. Your original classes
# -------------------------------------------------------------
class SeasonMapper:
    """Convert month → cropping season."""
    @staticmethod
    def month_to_season(df, month_col="Month"):
        df = df.copy()
        df["Season"] = None
        df.loc[df[month_col].between(6, 9), "Season"] = "Kharif"
        df.loc[(df[month_col] >= 10) | (df[month_col] <= 4), "Season"] = "Rabi"
        return df


class NDVIProcessor:
    """Convert monthly NDVI → seasonal NDVI."""
    @staticmethod
    def to_seasonal(ndvi_df):
        df = ndvi_df.copy()
        df["Year"] = df["Year"].astype(int)
        df["Month"] = df["Month"].astype(int)
        df = SeasonMapper.month_to_season(df, month_col="Month")
        seasonal = (
            df.groupby(["State", "Year", "Season"])["NDVI"]
              .mean()
              .reset_index()
              .rename(columns={"NDVI": "NDVI_SeasonalMean"})
        )
        return seasonal


class WeatherProcessor:
    @staticmethod
    def clean(weather_seasonal):
        df = weather_seasonal.copy()
        df["Year"] = df["Year"].astype(int)
        df["Season"] = df["Season"].astype(str)
        return df


class YieldProcessor:
    @staticmethod
    def clean(yield_df):
        df = yield_df.copy()
        df["Year"] = df["Year"].astype(int)
        df["Season"] = df["Season"].astype(str)
        return df


class SOCProcessor:
    @staticmethod
    def clean(soc_df):
        return soc_df.copy()


class DatasetMerger:
    def __init__(self, config=None):
        self.config = config

    def merge(self, weather_daily, weather_seasonal, ndvi, soc, yield_df):
        weather_seasonal = WeatherProcessor.clean(weather_seasonal)
        yield_df = YieldProcessor.clean(yield_df)
        soc = SOCProcessor.clean(soc)
        ndvi_seasonal = NDVIProcessor.to_seasonal(ndvi)

        merged = pd.merge(
            yield_df, weather_seasonal,
            on=["State", "Year", "Season"], how="left"
        )

        merged = pd.merge(
            merged, ndvi_seasonal,
            on=["State", "Year", "Season"], how="left"
        )

        merged = pd.merge(
            merged, soc,
            on="State", how="left"
        )

        merged = merged.dropna(subset=["Yield"])
        merged = merged.drop_duplicates()
        merged = merged.sort_values(["State", "Year", "Season"]).reset_index(drop=True)
        return merged


# -------------------------------------------------------------
# 2. Create sample input datasets
# -------------------------------------------------------------

# Example monthly NDVI data
ndvi_df = pd.DataFrame({
    "State": ["Punjab", "Punjab", "Punjab", "Punjab"],
    "Year": [2020, 2020, 2020, 2020],
    "Month": [6, 7, 10, 11],
    "NDVI": [0.30, 0.35, 0.25, 0.28]
})

# Seasonal weather data
weather_seasonal = pd.DataFrame({
    "State": ["Punjab", "Punjab"],
    "Year": [2020, 2020],
    "Season": ["Kharif", "Rabi"],
    "Rainfall": [500, 120],
    "Temp": [32, 18]
})

# SOC data (state-level)
soc = pd.DataFrame({
    "State": ["Punjab"],
    "SOC": [0.75]
})

# Yield data
yield_df = pd.DataFrame({
    "State": ["Punjab", "Punjab"],
    "Year": [2020, 2020],
    "Season": ["Kharif", "Rabi"],
    "Yield": [2200, 1800]
})

# -------------------------------------------------------------
# 3. Run the merger and display the result
# -------------------------------------------------------------
merger = DatasetMerger()

merged_df = merger.merge(
    weather_daily=None,       # Not used
    weather_seasonal=weather_seasonal,
    ndvi=ndvi_df,
    soc=soc,
    yield_df=yield_df
)

print("FINAL MERGED DATASET:")
print(merged_df)


FINAL MERGED DATASET:
    State  Year  Season  Yield  Rainfall  Temp  NDVI_SeasonalMean   SOC
0  Punjab  2020  Kharif   2200       500    32              0.325  0.75
1  Punjab  2020    Rabi   1800       120    18              0.265  0.75


In [2]:
!d:/Crop_yield_system/.jyp_env/Scripts/python.exe -m pip install pyyaml


  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl (154 kB)



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [45]:
import os
import sys

# Path to project root (one folder above notebooks/)
project_root = os.path.abspath("..")

# Add project root to PYTHONPATH
if project_root not in sys.path:
    sys.path.append(project_root)

project_root


'd:\\Crop_yield_system'

In [13]:
import os
import sys
import pandas as pd
import yaml
import seaborn as sns
import matplotlib.pyplot as plt

# ----------------------------------------
# Ensure project root is in PYTHONPATH
# ----------------------------------------
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

# Now safe to import src modules
from src.outlier_detection import OutlierHandler, ZScoreOutlierDetector, IQROutlierDetector

# Load config
config_path = os.path.join(project_root, "config.yaml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)


In [14]:
df = pd.read_csv("D:\\Crop_yield_system\\data\\Processed\\versions\\merged_20251124_153828.csv")
df.head()


,State,Year,Area (Hectare),Production (Tonnes),Yield (Tonne/Hectare),Season,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,ALLSKY_SFC_SW_DWN,RH2M,WS2M,NDVI_SeasonalMean,Mean_SOC,Median_SOC,Min_SOC,Max_SOC,Std_SOC
0,Chandigarh,2011,600,2725,4.54,Kharif,28.545820,32.636721,24.651230,1120.70,2415.25,73.597787,1.670820,0.505033,2.126695,2.0,2.0,6.25,0.384204
1,Chandigarh,2011,600,2725,4.54,Rabi,18.842547,26.398585,13.192453,81.17,3551.99,46.769575,1.713396,0.383363,2.126695,2.0,2.0,6.25,0.384204
2,Chandigarh,2012,575,2590,4.50,Kharif,30.871885,35.985164,26.062869,761.14,2549.54,58.650328,1.895164,0.457170,2.126695,2.0,2.0,6.25,0.384204
3,Chandigarh,2012,575,2590,4.50,Rabi,18.925164,26.774977,13.223897,87.20,3528.42,41.122770,1.888967,0.380036,2.126695,2.0,2.0,6.25,0.384204
4,Chandigarh,2013,575,2600,4.52,Kharif,28.820410,32.811475,24.909918,1174.10,2440.15,72.614426,1.535246,0.522958,2.126695,2.0,2.0,6.25,0.384204


In [15]:
outlier_cfg = config["preprocessing"]["outliers"]
handler = OutlierHandler(outlier_cfg)
detector = handler.detector
columns = handler.columns


In [16]:
columns


['Yield (Tonne/Hectare)',
 'NDVI_SeasonalMean',
 'T2M',
 'T2M_MAX',
 'T2M_MIN',
 'PRECTOTCORR',
 'ALLSKY_SFC_SW_DWN',
 'RH2M',
 'WS2M',
 'Mean_SOC',
 'Median_SOC',
 'Min_SOC',
 'Max_SOC',
 'Std_SOC']

In [17]:
mask = detector.detect(df, columns)
df_outliers = df[~mask]
df_clean_preview = df[mask]


In [24]:
len(df_outliers), len(df)
df.columns

Index(['State', 'Year', 'Area (Hectare)', 'Production (Tonnes)',
       'Yield (Tonne/Hectare)', 'Season', 'T2M', 'T2M_MAX', 'T2M_MIN',
       'PRECTOTCORR', 'ALLSKY_SFC_SW_DWN', 'RH2M', 'WS2M', 'NDVI_SeasonalMean',
       'Mean_SOC', 'Median_SOC', 'Min_SOC', 'Max_SOC', 'Std_SOC'],
      dtype='object')